# Initializing Variables

This section sets up the foundational components for the groundwater modeling workflow. Here, we define the modeling executables, project directories, and essential model parameters. This setup allows the user to control the structure and configuration of the model grid, including the cell size, number of layers, and the overall depth of the groundwater model.


## Importing Libraries

We begin by importing all necessary Python libraries required for the workflow, including scientific computing packages (like NumPy and Pandas), modeling packages (like FloPy), and geospatial libraries (like GeoPandas and Rasterio). All libraries are contained in the "common_imports" python file. This ensures that all necessary software tools are available within each notebook for subsequent processing.

In [1]:
#-----------------------Importing Libraries-----------------------#
# import os
# import flopy
# import pathlib
# import matplotlib.pyplot as plt
# import numpy as np
# import geopandas as gpd
# import pandas as pd
# import rasterio
# import pyproj
# import shutil
# import random
# import jupyter_book
# import pathlib as pl
# from pathlib import Path
# from rasterio.crs import CRS
# from rasterio.plot import show
# from rasterio.warp import calculate_default_transform, reproject, Resampling
# from rasterio.transform import from_bounds
# from rasterio.transform import rowcol
# from rasterio.mask import mask
# from shapely.geometry import box, Point, Polygon, LineString
# from flopy.utils.binaryfile import HeadFile
# from scipy.interpolate import griddata
# from pprint import pformat
# from flopy.plot.styles import styles
# from matplotlib.lines import Line2D
# from flopy.mf6 import MFSimulation
# from matplotlib import cbook, cm
# from matplotlib.colors import LightSource
# from modflow_devtools.misc import get_env, timed

%run /Users/u4eeevmq/Documents/Python/HyporheicFloPy/VQuintana/notebooks/common_imports.py


## Define Modeling Workspaces

In this section, we specify the file paths for the MODFLOW-6 and MODPATH-7 executables, which are used to run the groundwater flow and particle tracking simulations. We also define the main project workspace and subdirectories for organizing input files, model runs, and results. 

### Set Model Parameters

Model parameters are initialized for both the physical system and the numerical simulation. This includes specifying the length and time units, grid cell size, model depth, hydraulic properties (such as conductivity and porosity), as well as recharge rates and other optional settings. Users can customize these parameters to represent the physical characteristics of their specific study site or scenario.

In [2]:
#------------------------ Define Modeling Workspaces -----------------------#
# Modeling Executables
# Update these paths to your local MODFLOW-6 and MODPATH-7 executable paths
md6_exe_path = None
md7_exe_path = None

# Project workspace
sim_name = "Hyporheic_Project"
workspace = "HP_workspace"  # Default workspace directory 



In [3]:
#--------------------------Model Parameters-------------------------#
# Model units
length_units = "feet"
time_units = "days"

# Model Settings
nper = 1  # Number of stress periods
cell_size_x = cell_size_y = 5.0  # Grid cell size (10x10 feet) calculated from raster resolution
gw_mod_depth = 20.0  # Depth of the model (20 feet below the bed surface)
z = 0.5  # model layer thickness in feet
kh = 10.0  # Horizontal hydraulic conductivity (ft/day)
kv = 1.0  # Vertical hydraulic conductivity (ft/day)
gw_offset = 0.5  # Offset value (ft) for groundwater elevation (used to set initial head from surface water elevation)

# Optional Settings
porosity = 0.1 # Porosity
rch_iface = 6
rch_iflowface = -1
recharge_rate = 0.005  # Recharge rate ($ft/d$)

# Time discretization
nstp = 1
perlen = 1.0
tsmult = 1.0

In [4]:
# Parameters
md6_exe_path = "C:\\Users\\u4eeevmq\\Documents\\Python\\Flo_Py\\flopy\\modflowExe\\mf6.exe"
md7_exe_path = "C:\\Users\\u4eeevmq\\Documents\\Python\\Flo_Py\\flopy\\modflowExe\\mp7.exe"
sim_name = "Hyporheic_Project"
workspace = "HP_workspace"
length_units = "feet"
time_units = "days"
cell_size_x = 5.0
cell_size_y = 5.0
gw_mod_depth = 20.0
z = 0.5
kh = 10.0
kv = 1.0
gw_offset = 0.5
porosity = 0.1
rch_iface = 6
rch_iflowface = -1
recharge_rate = 0.005
nstp = 1
perlen = 1.0
tsmult = 1.0


## Workspace Directory Setup

To ensure a clean modeling environment, any pre-existing workspace is deleted before being recreated. The script then creates subdirectories for the groundwater flow and particle tracking models, and defines the names for model output files.

In [5]:
# Convert workspace to a Path object
workspace = Path(workspace)

# Clear the workspace directory if it exists
if os.path.exists(workspace):
    shutil.rmtree(workspace)   
os.makedirs(workspace)

# shorten model names so they fit in 16-char limit
gwf_name = "gwf_model"
mp7_name = "mp7_model"
gwf_ws = workspace / "gwf_workspace"
mp7_ws = workspace / "mp7_workspace"

# Create directories
workspace.mkdir(exist_ok=True, parents=True)
gwf_ws.mkdir(exist_ok=True, parents=True)
mp7_ws.mkdir(exist_ok=True, parents=True)

# Define output file names
# Head Files
headfile = "{}.hds".format(gwf_name)
head_filerecord = [headfile]

# Budget Files
budgetfile = "{}.cbb".format(gwf_name)
budget_filerecord = [budgetfile]

# Settings from environment variables
write = get_env("WRITE", True)
run = get_env("RUN", True)
plot = get_env("PLOT", True)
plot_show = get_env("PLOT_SHOW", True)
plot_save = get_env("PLOT_SAVE", True)

## Storing Variables for Reproducibility

All important variables and parameters are stored using the `%store` magic command. This facilitates sharing between notebooks.

In [6]:
# Store variables
%store md6_exe_path
%store md7_exe_path
%store sim_name
%store workspace
%store gwf_name
%store mp7_name
%store gwf_ws
%store mp7_ws
%store headfile
%store head_filerecord
%store budgetfile
%store budget_filerecord
%store write
%store run
%store plot
%store plot_show
%store plot_save

# Store model parameters
%store length_units
%store time_units
%store nper
%store cell_size_x
%store cell_size_y
%store gw_mod_depth
%store z
%store kh
%store kv
%store gw_offset
%store porosity
%store rch_iface
%store rch_iflowface
%store recharge_rate
%store nstp
%store perlen
%store tsmult

Stored 'md6_exe_path' (str)
Stored 'md7_exe_path' (str)
Stored 'sim_name' (str)
Stored 'workspace' (WindowsPath)
Stored 'gwf_name' (str)
Stored 'mp7_name' (str)
Stored 'gwf_ws' (WindowsPath)
Stored 'mp7_ws' (WindowsPath)
Stored 'headfile' (str)
Stored 'head_filerecord' (list)
Stored 'budgetfile' (str)
Stored 'budget_filerecord' (list)
Stored 'write' (bool)
Stored 'run' (bool)
Stored 'plot' (bool)
Stored 'plot_show' (bool)
Stored 'plot_save' (bool)
Stored 'length_units' (str)
Stored 'time_units' (str)
Stored 'nper' (int)
Stored 'cell_size_x' (float)
Stored 'cell_size_y' (float)
Stored 'gw_mod_depth' (float)
Stored 'z' (float)
Stored 'kh' (float)
Stored 'kv' (float)
Stored 'gw_offset' (float)
Stored 'porosity' (float)
Stored 'rch_iface' (int)
Stored 'rch_iflowface' (int)
Stored 'recharge_rate' (float)
Stored 'nstp' (int)
Stored 'perlen' (float)
Stored 'tsmult' (float)
